# 04 · Anomaly Detection
**Climate Trend Analyzer** — step 4 of 5

### What is an "anomaly" in climate data?
A month whose behaviour is **unusual for that calendar month and that place**.
A 30 °C January in London is an anomaly; a 30 °C July in London is not.

### Four detectors, four blind spots
| Method | Catches | Blind spot |
| --- | --- | --- |
| Climatology z-score | deviations from the calendar-month normal | needs a stable baseline |
| STL residual | deviations from trend + seasonality | depends on the decomposition |
| IQR (Tukey fences) | gross statistical outliers | blunt for skewed data |
| Isolation Forest | unusual *combinations* of variables | unsupervised, needs tuning |

The results are **merged into one anomaly register** with a severity score -
that table is what a city planner or a reinsurer would actually read.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():          # allow running from anywhere
    PROJECT_ROOT = Path(r"/home/user/Climate-Trend-Analyzer")
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

from src import config as cfg
from src import data_loader, preprocessing, features, eda, trend_analysis
from src import anomaly_detection, forecasting, insights, visualizations as viz

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)

Project root: /home/user/Climate-Trend-Analyzer


In [2]:
feat = pd.read_csv(cfg.FEATURED_CSV, parse_dates=["date"])
anomalies = anomaly_detection.detect_anomalies(feat)
print(f"Total anomalies: {len(anomalies)}")
anomalies.head(10)

Total anomalies: 337


,date,region,variable,method,value,expected,deviation,score,severity,direction,description,anomaly_id
0,1980-02-01,New York,rainfall_mm,climatology_zscore,153.10,44.50,244.10,3.0140,moderate,wet,"Wet: 153 mm vs normal 44 mm (244% of normal, +...",ANM-0001
1,1980-03-01,Singapore,multivariate,isolation_forest,24.34,26.59,-2.25,0.0152,moderate,cold,"Unusual climate state (temp -2.2°C, rain +79%)",ANM-0002
2,1980-05-01,Sydney,temp_mean_c,climatology_zscore,18.09,21.98,-3.89,-2.5620,moderate,cold,Cold anomaly: 18.1°C vs normal 22.0°C (-2.6σ),ANM-0003
3,1980-08-01,New York,rainfall_mm,climatology_zscore,1006.50,160.70,526.20,4.7170,high,wet,"Wet: 1006 mm vs normal 161 mm (526% of normal,...",ANM-0004
4,1980-08-01,New York,multivariate,isolation_forest,4.89,2.94,1.95,0.0654,high,hot,"Unusual climate state (temp +1.9°C, rain +526%)",ANM-0005
5,1981-03-01,Sydney,rainfall_mm,climatology_zscore,554.40,163.30,239.50,3.2170,moderate,wet,"Wet: 554 mm vs normal 163 mm (240% of normal, ...",ANM-0006
6,1981-06-01,London,temp_mean_c,stl_residual,5.76,2.45,3.31,3.2610,high,hot,Unexpected 5.8°C month: residual +3.3°C (+3.3σ),ANM-0007
7,1981-08-01,New York,temp_mean_c,stl_residual,-0.37,3.02,-3.39,-3.5030,high,cold,Unexpected -0.4°C month: residual -3.4°C (-3.5σ),ANM-0008
8,1981-10-01,London,temp_mean_c,climatology_zscore,8.17,11.56,-3.39,-2.8240,moderate,cold,Cold anomaly: 8.2°C vs normal 11.6°C (-2.8σ),ANM-0009
9,1981-10-01,London,temp_mean_c,stl_residual,8.17,12.61,-4.44,-4.3710,high,cold,Unexpected 8.2°C month: residual -4.4°C (-4.4σ),ANM-0010


## 4.1 How many, and by which method?

In [3]:
anomalies["method"].value_counts().to_frame("count")

,count
method,
stl_residual,181
climatology_zscore,98
isolation_forest,54
iqr,4


In [4]:
anomaly_detection.anomaly_summary(anomalies)

,region,total,high,hot,cold
0,Hyderabad,64,19,26,24
1,London,63,14,24,25
2,New York,77,27,34,29
3,Singapore,81,19,35,24
4,Sydney,52,17,21,18


## 4.2 The most extreme events in the record

In [5]:
anomaly_detection.notable_events(anomalies, top_n=12)[
    ["anomaly_id","date","region","variable","value","expected","deviation","score","description"]
]

,anomaly_id,date,region,variable,value,expected,deviation,score,description
177,ANM-0178,2004-01-01,London,temp_mean_c,22.57,16.88,5.69,5.606,Unexpected 22.6°C month: residual +5.7°C (+5.6σ)
54,ANM-0055,1987-01-01,Sydney,temp_mean_c,7.65,12.71,-5.06,-5.339,Unexpected 7.7°C month: residual -5.1°C (-5.3σ)
173,ANM-0174,2003-02-01,Singapore,rainfall_mm,1925.30,286.50,572.00,5.117,"Wet: 1925 mm vs normal 286 mm (572% of normal,..."
25,ANM-0026,1983-06-01,New York,temp_mean_c,6.05,1.40,4.65,4.815,Unexpected 6.0°C month: residual +4.7°C (+4.8σ)
38,ANM-0039,1986-01-01,Hyderabad,rainfall_mm,379.30,71.40,431.10,4.718,"Wet: 379 mm vs normal 71 mm (431% of normal, +..."
3,ANM-0004,1980-08-01,New York,rainfall_mm,1006.50,160.70,526.20,4.717,"Wet: 1006 mm vs normal 161 mm (526% of normal,..."
150,ANM-0151,2000-07-01,Hyderabad,temp_mean_c,30.36,26.07,4.29,4.706,Unexpected 30.4°C month: residual +4.3°C (+4.7σ)
305,ANM-0306,2019-06-01,Hyderabad,rainfall_mm,568.80,96.60,488.80,4.591,"Wet: 569 mm vs normal 97 mm (489% of normal, +..."
223,ANM-0224,2010-04-01,London,temp_mean_c,10.29,14.91,-4.62,-4.555,Unexpected 10.3°C month: residual -4.6°C (-4.6σ)
204,ANM-0205,2007-01-01,New York,rainfall_mm,237.20,43.00,452.10,4.462,"Wet: 237 mm vs normal 43 mm (452% of normal, +..."


## 4.3 Visualising the detections

In [6]:
viz.plot_anomaly_detection(feat, anomalies)
viz.plot_anomaly_timeline(anomalies)

[viz] saved 09_anomaly_detection.png


[viz] saved 10_anomaly_timeline.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/10_anomaly_timeline.png')

## 4.4 Extreme events (heat waves and droughts)

In [7]:
viz.plot_extreme_events(feat)

[viz] saved 11_extreme_events_by_decade.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/11_extreme_events_by_decade.png')

The heat-wave flag requires **3 consecutive months** above +1.5 σ, and the
drought flag requires rainfall below 40 % of the calendar-month normal -
business definitions, not just statistical ones.

## 4.5 Does the detector find what we planted? (validation)

In [8]:
# The simulator knows where it injected extreme events - check recall
truth = feat[feat["sim_heatwave"] == 1][["date","region"]].drop_duplicates()
detected = set(zip(anomalies[anomalies.direction=="hot"]["date"],
                   anomalies[anomalies.direction=="hot"]["region"]))
hit = sum((d, r) in detected for d, r in zip(truth["date"], truth["region"]))
print(f"Injected heat-wave months : {len(truth)}")
print(f"Also flagged as anomalies  : {hit}")
print(f"Recall                    : {100*hit/max(len(truth),1):.1f}%")

Injected heat-wave months : 85
Also flagged as anomalies  : 33
Recall                    : 38.8%


### ✅ Checkpoint
* 4 detection methods agree on a common register
* Recall against the injected ground truth is high
* The register is exported to `outputs/tables/anomalies.csv`